# Install libraries

In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install chromadb
!pip install sentence-transformers
!pip install pypdf

# Imports

In [2]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Define Paths

In [3]:
PROJECT_DIR = Path("..").resolve()

DATA_DIR = PROJECT_DIR / "data"
VECTOR_DB_DIR = PROJECT_DIR / "vector_db"

print("Data folder:", DATA_DIR)
print("Vector DB:", VECTOR_DB_DIR)

Data folder: C:\Users\raina\bns-rag-chatbot\data
Vector DB: C:\Users\raina\bns-rag-chatbot\vector_db


# Load PDFs

In [4]:
documents = []

for pdf in DATA_DIR.glob("*.pdf"):
    loader = PyPDFLoader(str(pdf))
    docs = loader.load()
    documents.extend(docs)

print("Total pages loaded:", len(documents))

Total pages loaded: 351


# Chunk Documents

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 1966


# Load Embedding Model

In [6]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


# Create Vector Database

In [7]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=str(VECTOR_DB_DIR),
    collection_name="bns_collection"
)

vectorstore.persist()

print("Vector database created")


Vector database created


C:\Users\raina\AppData\Local\Temp\ipykernel_16284\2929056899.py:8: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


# Create Retriever

In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

print("Retriever ready")

Retriever ready


# Ask Question

In [9]:
query = "What is punishment for murder?"

docs = retriever.invoke(query)

for i,doc in enumerate(docs):
    print("\nResult",i+1)
    print(doc.page_content[:500])


Result 1
years, and shall also be liable to fine.
109.(1) Whoever does any act with such intention or knowledge, and under such
circumstances that, if he by that act caused death, he would be guilty of murder, shall be
punished with imprisonment of either description for a term which may extend to ten years,
and shall also be liable to fine; and if hurt is caused to any person by such act, the offender
shall be liable either to imprisonment for life, or to such punishment as is hereinbefore
mentioned.
Cu

Result 2
years, and shall also be liable to fine.
109.(1) Whoever does any act with such intention or knowledge, and under such
circumstances that, if he by that act caused death, he would be guilty of murder, shall be
punished with imprisonment of either description for a term which may extend to ten years,
and shall also be liable to fine; and if hurt is caused to any person by such act, the offender
shall be liable either to imprisonment for life, or to such punishment as is herei

In [10]:
!pip install transformers accelerate

# LLM generation

## TinyLlama

In [20]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

D:\Python_anoconda\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\raina\.cache\huggingface\hub\models--Qwen--Qwen2-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

# Generation function

In [54]:
def ask_bns(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([
        d.page_content[:250] for d in docs
    ])

    prompt = f"""
You are a legal assistant for Bharatiya Nyaya Sanhita (BNS).

Answer clearly using the legal context.
Limit the response to 3–4 complete sentences.

Context:
{context}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=90,
        do_sample=False,
        early_stopping=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = generated_text.split("Answer:")[-1].strip()

    if not answer.endswith("."):
        answer += "."

    return answer

# Testing it

In [55]:
ask_bns("What is punishment for murder under BNS?")

'Under the Indian Penal Code (IPC), murder is punishable by death or imprisonment for life. The IPC provides for different punishments for various crimes, including murder. In general, murder can be committed in several ways, including:\n\n1. Murder by force: This includes killing someone intentionally through physical violence, whether accidental or intentional.\n\n2. Murder by accident: This involves unintentional killing, where the victim is not aware that they have been killed.\n\n3.'

In [56]:
ask_bns("What is the punishment for theft?")

'The punishment for theft is imprisonment of either description for a term which may extend to seven years, and also liability to fine.'

In [57]:
ask_bns("What section describes culpable homicide?")

'Culpable homicide is described in Section 102 of the Indian Penal Code. This section outlines the circumstances under which an individual can be held liable for committing culpable homicide. It specifies that if a person intentionally causes the death of another person, whether intentional or not, the person is guilty of culpable homicide. The section also mentions that the death must be caused by the act of the person who committed the crime and that the person cannot.'